# Surface Topography

This notebook is used to plot the surface topography

# Plot Topography from ASPECT outputs

In [ ]:
from pathlib import Path
import os, sys
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import rcdefaults, gridspec
from matplotlib.ticker import MultipleLocator
import pyvista as pv

# Include this pakage
HaMaGeoLib_DIR = "/home/lochy/ASPECT_PROJECT/HaMaGeoLib"
if os.path.abspath(HaMaGeoLib_DIR) not in sys.path:
    sys.path.append(os.path.abspath(HaMaGeoLib_DIR))
from hamageolib.utils.exception_handler import my_assert
import hamageolib.utils.plot_helper as plot_helper
from hamageolib.research.haoyuan_collision0.post_process import integrate_topography

# # Retrieve the default color cycle
default_colors = [color['color'] for color in plt.rcParams['axes.prop_cycle']]

# Plot options
scaling_factor = 1.0  # scale factor of plot
font_scaling_multiplier = 1.5 # extra scaling multiplier for font
legend_font_scaling_multiplier = 0.5
line_width_scaling_multiplier = 2.0 # extra scaling multiplier for lines
n_minor_ticks = 4  # number of minor ticks between two major ones

# scale the matplotlib params
plot_helper.scale_matplotlib_params(scaling_factor, font_scaling_multiplier=font_scaling_multiplier,\
                        legend_font_scaling_multiplier=legend_font_scaling_multiplier,
                        line_width_scaling_multiplier=line_width_scaling_multiplier)

# Update font settings for compatibility with publishing tools like Illustrator.
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'pdf.fonttype': 42,
    'ps.fonttype': 42})

In [ ]:
plot_aspect_topo_output = False

if plot_aspect_topo_output:

    # Also plot the topography average
    plot_topography_at_time_with_average = True
    
    
    output_dir = "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026"
    topography_files = [
        # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR/output_continent_ocean_continent/topography/topography.00008",
        # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR/output_continent_ocean_continent/topography/topography.00009",
        # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR/output_continent_ocean_continent/topography/topography.00010"
        # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR_anne/output_continent_ocean_continent/topography/topography.00008",
        # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR_anne/output_continent_ocean_continent/topography/topography.00009",
        # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR_anne/output_continent_ocean_continent/topography/topography.00010"
        # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR_no_ini/output_continent_ocean_continent/topography/topography.00008",
        # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR_no_ini/output_continent_ocean_continent/topography/topography.00009",
        # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR_no_ini/output_continent_ocean_continent/topography/topography.00010"
        "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR_no_ini_anne/output_continent_ocean_continent/topography/topography.00008",
        "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR_no_ini_anne/output_continent_ocean_continent/topography/topography.00009",
        "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/FS_lowR_no_ini_anne/output_continent_ocean_continent/topography/topography.00010"
                        ]
    
    
    
    # create the plot
    fig, ax = plt.subplots(figsize=(10, 6))
    # ax_twinx = ax.twinx()
    
    # options of plot
    x_lim = [0, 5500]
    # x_lim = [0, 5500]; x_half_width = None  # km, options for plotting the whole width
    # x_lim = None; x_half_width = 500 # km, options for plotting around the suture
    topo_lim = [-5000, 5000]  # m
    topo_tick_interval = 500.0
    deformation_ratio_lim = [-0.5, 0.5]
    deformation_ratio_tick_interval = 0.1
    
    
    for i, topography_file in enumerate(topography_files):
    
        # Extract topography data
        my_assert(os.path.isfile(topography_file), FileExistsError, "%s doesn't exist." % topography_file)
    
        print("Read from %s" % topography_file)
        data = np.loadtxt(topography_file, comments="#")
    
        x = data[:, 0]
        topography = data[:, 2]
    
        _, average_topography = integrate_topography(x, topography)
        
        # creat the plot
        ax.plot(x/1e3, topography, label="topography %d" % i, color=default_colors[i])
    
        if plot_topography_at_time_with_average:
            ax.hlines(
                average_topography,
                xmin=x_lim[0],
                xmax=x_lim[1],
                color=default_colors[i],
                linestyle=":",
            )
    
            ax.annotate(
                f"Average: {average_topography:.0f} m",
                xy=(x_lim[1], average_topography),
                xytext=(-5, 5),
                textcoords="offset points",
                ha="right",
                color=default_colors[i],
            )
    
    
    # adjust plot axis
    ax.set_xlabel("X (km)")
    ax.set_xlim(x_lim)
    
    ax.set_ylabel("Topography (m)") # , color=default_colors[0])
    ax.set_ylim(topo_lim)
    ax.yaxis.set_major_locator(MultipleLocator(topo_tick_interval))
    ax.yaxis.set_minor_locator(MultipleLocator(topo_tick_interval/(n_minor_ticks+1)))
    
    ax.legend()
    
    ax.grid()
    
    fig_path = os.path.join(output_dir, "topography")
    fig.savefig(fig_path + ".png")
    print("Saved figure %s" % (fig_path + ".png"))
    
    fig.savefig(fig_path + ".pdf")
    print("Saved figure %s" % (fig_path + ".pdf"))
    

# Plot Results from FastScape

In [ ]:
plot_results_from_fastscape = True

if plot_results_from_fastscape:

    # Compare the y-averaged topography before and after the standalone FastScape timestep-9 replay.
    from pathlib import Path
    import numpy as np
    from matplotlib import pyplot as plt
    
    fastscape_case_dir = Path(
        "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test_10082026/"
        "FS_lowR_no_ini_coupling_dump"
    )
    fastscape_input_file = (
        fastscape_case_dir
        / "output_continent_ocean_continent/fastscape/coupling-input-0000009.txt"
    )
    fastscape_result_dir = fastscape_case_dir / "fastscape_standalone_timestep_9"
    fastscape_result_file = fastscape_result_dir / "final_topography.txt"
    
    if not fastscape_input_file.is_file():
        raise FileNotFoundError(f"FastScape input was not found: {fastscape_input_file}")
    if not fastscape_result_file.is_file():
        raise FileNotFoundError(f"FastScape result was not found: {fastscape_result_file}")
    
    metadata = {}
    with fastscape_input_file.open() as stream:
        for line in stream:
            if not line.startswith("#"):
                break
            fields = line[1:].split(maxsplit=1)
            if len(fields) == 2:
                metadata[fields[0]] = fields[1]
    
    nx = int(metadata["nx"])
    ny = int(metadata["ny"])
    fastscape_iterations = int(metadata["fastscape_iterations"])
    fastscape_dt = float(metadata["fastscape_timestep_years"])
    
    input_data = np.loadtxt(fastscape_input_file, comments="#")
    final_topography = np.loadtxt(fastscape_result_file)
    if input_data.shape != (nx * ny, 11):
        raise ValueError(f"Expected {(nx * ny, 11)} input values, got {input_data.shape}.")
    if final_topography.size != nx * ny:
        raise ValueError(f"Expected {nx * ny} final elevations, got {final_topography.size}.")
    
    x_grid = input_data[:, 1].reshape(ny, nx)
    initial_grid = input_data[:, 3].reshape(ny, nx)
    final_grid = final_topography.reshape(ny, nx)
    if not np.allclose(x_grid, x_grid[0]):
        raise ValueError("The x coordinates differ between FastScape y rows.")
    
    x_km = x_grid[0] / 1e3
    initial_mean = initial_grid.mean(axis=0)
    final_mean = final_grid.mean(axis=0)
    
    # make the plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(x_km, initial_mean, label="Before FastScape timestep 9")
    ax.plot(
        x_km,
        final_mean,
        label=f"After {fastscape_iterations} steps ({fastscape_iterations * fastscape_dt / 1e3:.1f} kyr)",
    )
    ax.set_xlabel("X (km)")
    ax.set_ylabel("Y-averaged topography (m)")
    ax.set_title("Standalone FastScape replay of ASPECT timestep 9")
    ax.grid()
    ax.legend()
    fig.tight_layout()
    
    figure_path = fastscape_result_dir / "average_topography_profile"
    fig.savefig(figure_path.with_suffix(".png"), dpi=200)
    fig.savefig(figure_path.with_suffix(".pdf"))
    print(f"Initial y-averaged range: {initial_mean.min():.2f} to {initial_mean.max():.2f} m")
    print(f"Final y-averaged range: {final_mean.min():.2f} to {final_mean.max():.2f} m")
    print(f"Saved {figure_path.with_suffix('.png')}")
    
    

In [ ]:
# reset plot options
rcdefaults()